In [1]:
!pip install -q langchain langchain-groq
!pip install -q chromadb

In [88]:
#llm creating
import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.2, max_retries=3)

In [73]:
#creating tools
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}
HOURS = {"monday": "9 am to 8 pm", "tuesday": "9 am to 8 pm", "wednesday": "9 am to 8 pm",
         "thursday": "9 am to 8 pm", "friday": "9 am to 8 pm", "saturday": "9 am to 8 pm",
         "sunday": "10 am to 6 pm"}


class Order(BaseModel):
    """A cake order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["plum cake", "chocolate cake", "cupcake"] = Field(description="Menu item")
    qty: int = Field(description="How many", ge=1)
    date: str = Field(description="Pickup date as the customer said it")
    eggless: bool = Field(description="True if the customer asked for eggless")

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price in rupees for a quantity of one menu item.

    Args:
        item: Menu item, e.g. "plum cake", "chocolate cake" or "cupcake"
        qty: How many the customer wants
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the bakery's opening hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a confirmed cake order to the Orders sheet.
    Use only when the customer has given their name, item, quantity and pickup date."""
    try:
        order = Order(name=name, item=item.lower().strip(), qty=qty, date=date, eggless=eggless)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = MENU[order.item] * order.qty
    ws.append_row([order.name, order.item, order.qty, order.date,
                   "yes" if order.eggless else "no", total])
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}, Rs {total}."

@tool
def tool1(item: str) -> str:
    """Does stuff."""
    return f"{item} your query is not clear please try again"

In [92]:
#creating vetordatabase with help of embedings and storing
import chromadb

FAQS = {
    "delivery":     "We deliver within 5 km of the bakery. Delivery is Rs 50, free for orders above Rs 1,000.",
    "payment":      "We accept UPI and cash. Orders above Rs 1,000 need a 50% advance by UPI.",
    "cancellation": "Cancellation is free up to 12 hours before pickup. After that, the advance is not refunded.",
    "photo_cake":   "Photo cakes and custom designs need 48 hours' notice.",
    "notice":       "Regular orders need 24 hours' notice.",
    "eggless":      "All our cakes have eggless versions at the same price.",
    "allergens":    "Plum cake contains nuts and dried fruit. Cupcakes and chocolate cake are nut-free.",
    "storage":      "Keep cakes in the fridge and eat them within 3 days.",
    "message":      "A message on the cake is free, up to 20 letters.",
    "candles":      "Candles are free with every cake.",
    "discounts":    "Bulk-order discounts need the owner's approval.",
    "pickup":       "Pickup is during opening hours: Monday to Saturday 9 am to 8 pm, Sunday 10 am to 6 pm.",
    "birthday_cake":"Birthday cakes can be ordered in different flavours and designs.",
    "custom_design":"Custom cake designs need to be confirmed with the bakery before ordering.",
    "order_changes": "Order changes should be requested as early as possible and confirmed by the bakery.",
    "freshness": "Cakes should be refrigerated and consumed within 3 days.",
    "delivery_area": "Delivery is available within 5 km of the bakery.",
    "advance_payment": "Orders above Rs 1,000 require a 50% advance payment through UPI."

}

client = chromadb.Client()
faq = client.get_or_create_collection(name="sweet_crumbs_faq")
faq.upsert(ids=list(FAQS.keys()), documents=list(FAQS.values()))   # upsert = add or update
print(faq.count(), "facts stored")
print(faq.query(query_texts=["Can you send the cake to my house?"], n_results=2)["documents"][0])

19 facts stored
['A message on the cake is free, up to 20 letters.', 'Cakes should be refrigerated and consumed within 3 days.']


In [63]:
#The search_faq tool
from langchain.tools import tool
@tool
def search_faq(question: str) -> str:
    """Search Sweet Crumbs' policies and FAQs: delivery, payment, cancellation,
    photo cakes, allergens, storage, candles and messages on cakes.
    Use for any question about rules or policies, not for prices or opening hours."""
    results = faq.query(query_texts=[question], n_results=3)
    return "\n".join(results["documents"][0])

In [91]:
# The agent with memory

from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
RULES:  Use get_price for prices and is_open for timings; never guess them.
        For any policy question, use search_faq and answer ONLY from what it returns.
        If the answer is not there, must tell that "check with the owner."
        If the answer isn't there, must say that "Check with the owner."
        Before log_order, make sure you have name, item, quantity and pickup date; ask if missing.
FORMAT: Under 40 words, friendly, English or Telugu matching the customer."""

agent = create_agent(model=llm, tools=[get_price, is_open, log_order, search_faq],
                     system_prompt=SYSTEM, checkpointer=InMemorySaver())

def chat(thread_id: str, text: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": text}]}, config)
        return result["messages"][-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print(chat("ravi", "Hi, I'm Ravi. Do you deliver? My order is about Rs 1,300."))
print(chat("ravi", "Does the plum cake have nuts?"))
print(chat("sravani", "Can I cancel tomorrow morning if I pick up tomorrow evening?"))
print(chat("ravi", "What's my name, and will I pay for delivery?"))
# memory + RAG together

Hi Ravi! Yes, we deliver within 5 km. Delivery is ₹50, but it’s free for orders over ₹1,000—so yours qualifies. Just give us 24 hours notice. Let us know what you’d like to order!
Yes, our plum cake contains nuts (and dried fruit). Let me know if you’d like to order it!
Sorry, you can’t cancel for free that close—cancellations are free only up to 12 hours before pickup. After that, the advance isn’t refunded. Check with the owner for any exceptions.
Your name is Ravi, and delivery will be free since your order is over ₹1,000.


In [105]:

import csv
import re
import unicodedata

def clean(text):
    text = unicodedata.normalize("NFKC", text).casefold()
    return re.sub(r"[^\w\s]", "", text).strip()

def run_tests(path="/content/tests.csv"):
    with open(path, mode="r", encoding="latin1") as f:
        rows = list(csv.DictReader(f))

    passed = 0

    for i, row in enumerate(rows):
        question = row["questions"].strip()
        expected = clean(row["must_include"])

        thread_id = f"harsha{i}"
        answer = chat(thread_id, question)

        clean_answer = clean(answer)
        ok = expected in clean_answer

        if ok:
            passed += 1

        print(
            "PASS" if ok else "FAIL",
            "|", question,
            "→", answer[:110]
        )

        if not ok:
            print("Expected:", expected)
            print("Actual:", clean_answer[:200])

    total = len(rows)

    if total:
        print(f"\n{passed}/{total} passed ({passed / total:.0%})")
    else:
        print("\nNo test cases found.")

run_tests()


PASS | How much notice do regular cake orders need? → Regular cake orders need at least 24 hours’ notice. Happy baking!
PASS | Are candles free with a cake? → Yes—candles are free with every cake you order! 🎉
PASS | How many letters can I put in a cake message? → A free cake message can contain up to 20 letters. 🎂
PASS | How should I store the cake? → Keep the cake refrigerated and enjoy it within 3 days for the best taste and freshness.
PASS | How long should I keep the cake before eating it? → Keep it refrigerated and enjoy within 3 days.
PASS | When can I cancel for free? → Free cancellation is allowed up to 12 hours before your scheduled pickup. After that, the advance isn’t refund
PASS | Do you have eggless cakes? → Yes! All our cakes are available in egg‑less versions at the same price. Let us know which one you’d like!
PASS | Are cupcakes nut-free? → Yes—our cupcakes contain no nuts. 🎂✨Yes—our cupcakes are nut‑free. 🎂✨
FAIL | Is chocolate cake nut-free? → Yes, our chocolate cake